# 43 — Chứng quyền có bảo đảm

Chứng quyền là nơi cái bẫy đơn vị của thị trường Việt Nam lộ ra rõ nhất: **giá
cổ phiếu tính bằng nghìn VND, giá chứng quyền tính bằng VND thô**. Ghép hai thứ
đó mà không quy đổi là sai đúng **1000 lần**, và không có exception nào.

Notebook này:

1. Dựng lại cái bẫy 1000 lần, và cách chặn nó
2. Danh mục 311 chứng quyền đang lưu hành
3. Moneyness — và **một giới hạn của dữ liệu phải nói thẳng**
4. Thanh khoản: phần lớn chứng quyền gần như không giao dịch

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, duong, hom_nay, lui_ngay, nhan_don_vi
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

## 1 · ⚠️ Cái bẫy 1000 lần

Dựng lại nó bằng dữ liệu thật, rồi chỉ ra chỗ nó lộ mặt.

In [2]:
CO_SO = "HPG"

cw_hpg = client.meta.warrants(underlying=CO_SO)
MA_CW = cw_hpg["symbol"].iloc[0]

gia_cp = client.eod.stock.ohlcv(CO_SO, start=lui_ngay(HOM_NAY, thang=1))
gia_cw = client.eod.warrant.ohlcv(MA_CW, start=lui_ngay(HOM_NAY, thang=1))

print(f"{CO_SO:<10} close = {gia_cp['close'].iloc[-1]:>10,.2f}   đơn vị = {gia_cp.attrs['finlens']['units']['close']}")
print(f"{MA_CW:<10} close = {gia_cw['close'].iloc[-1]:>10,.2f}   đơn vị = {gia_cw.attrs['finlens']['units']['close']}")
print(f"\nTỷ lệ giữa hai con số: {gia_cw['close'].iloc[-1] / gia_cp['close'].iloc[-1]:,.0f} lần")
print("→ nhìn qua tưởng chứng quyền đắt gấp 70 lần cổ phiếu. Thực ra là hai đơn vị khác nhau.")

HPG        close =      22.10   đơn vị = kVND
CHPG2525   close =   1,630.00   đơn vị = VND

Tỷ lệ giữa hai con số: 74 lần
→ nhìn qua tưởng chứng quyền đắt gấp 70 lần cổ phiếu. Thực ra là hai đơn vị khác nhau.


In [3]:
# Cách SAI — ghép rồi tính, attrs bị xoá và không còn gì cảnh báo
ghep_sai = pd.concat([gia_cp, gia_cw])
print(f"attrs sau pd.concat: {ghep_sai.attrs}")
print("\nGiá trung bình 'của cả hai mã':")
print(ghep_sai.groupby("symbol", observed=True)["close"].mean().round(2).to_string())
print("\n→ Hai con số này không cộng, không trừ, không trung bình được với nhau.")

attrs sau pd.concat: {}

Giá trung bình 'của cả hai mã':
symbol
CHPG2525    1983.18
HPG           21.65

→ Hai con số này không cộng, không trừ, không trung bình được với nhau.


In [4]:
# Cách ĐÚNG — quy về một đơn vị TRƯỚC khi ghép, và đặt tên nói rõ đơn vị
def ve_vnd(df: pd.DataFrame, cot: str = "close") -> pd.Series:
    """Quy cột giá về VND dựa trên đơn vị FinLens khai, không dựa vào trí nhớ."""
    don_vi = (df.attrs.get("finlens") or {}).get("units", {}).get(cot)
    he_so = {"kVND": 1_000, "VND": 1}.get(don_vi)
    if he_so is None:
        raise ValueError(f"Không biết quy đổi đơn vị {don_vi!r} sang VND")
    return df[cot] * he_so


ghep_dung = pd.concat(
    [
        gia_cp.assign(close_vnd=ve_vnd(gia_cp)),
        gia_cw.assign(close_vnd=ve_vnd(gia_cw)),
    ]
)
print("Sau khi quy về VND:")
print(ghep_dung.groupby("symbol", observed=True)["close_vnd"].last().round(0).to_string())

Sau khi quy về VND:
symbol
CHPG2525     1630.0
HPG         22100.0


Hàm `ve_vnd` **đọc đơn vị từ `attrs`** rồi mới nhân. Nếu FinLens đổi đơn vị của
một namespace nào đó, hàm này ném `ValueError` thay vì âm thầm ra số sai. Đó là
khác biệt giữa "nhân 1000 vì tôi nhớ là nghìn đồng" và "nhân 1000 vì frame nói
nó là nghìn đồng".

## 2 · Danh mục chứng quyền

`meta.warrants()` mặc định chỉ trả về các mã **còn hiệu lực**.

In [5]:
con_han = client.meta.warrants()
tat_ca = client.meta.warrants(active_only=False)

print(f"Còn hiệu lực : {len(con_han):>4} mã")
print(f"Kể cả đã đáo hạn: {len(tat_ca):>4} mã")
print(f"\nCột: {list(con_han.columns)}")
print(f"Đơn vị: {[f'{k}={v}' for k, v in con_han.attrs['finlens']['units'].items() if v]}")
con_han.head(5)

Còn hiệu lực :  311 mã
Kể cả đã đáo hạn:  720 mã

Cột: ['symbol', 'underlying', 'issuer', 'exercise_price', 'maturity_date']
Đơn vị: ['exercise_price=VND']


,symbol,underlying,issuer,exercise_price,maturity_date
0,CACB2511,ACB,SSI,19832.0,2026-09-23
1,CACB2514,ACB,VND,23540.0,2026-09-08
2,CACB2515,ACB,VND,27420.0,2027-03-08
3,CACB2516,ACB,KAI,24575.0,2026-10-16
4,CACB2517,ACB,KAI,25006.0,2027-01-18


In [6]:
print(f"{con_han['underlying'].nunique()} mã cơ sở · {con_han['issuer'].nunique()} tổ chức phát hành\n")

theo_ph = con_han["issuer"].value_counts().reset_index()
theo_ph.columns = ["tổ chức phát hành", "số chứng quyền"]

bar_ngang(
    theo_ph,
    nhan="tổ chức phát hành",
    gia_tri="số chứng quyền",
    tieu_de="Số chứng quyền đang lưu hành theo tổ chức phát hành",
    phu_de="Tổ chức phát hành là bên chịu nghĩa vụ thanh toán — không phải doanh nghiệp cơ sở",
    nhan_x="số mã",
    dinh_dang_nhan="{:.0f}",
)

21 mã cơ sở · 12 tổ chức phát hành



In [7]:
theo_cs = con_han["underlying"].value_counts().reset_index()
theo_cs.columns = ["mã cơ sở", "số chứng quyền"]

bar_ngang(
    theo_cs,
    nhan="mã cơ sở",
    gia_tri="số chứng quyền",
    tieu_de="Số chứng quyền theo mã cơ sở",
    phu_de="Chỉ những mã vốn hoá lớn, thanh khoản cao mới có chứng quyền",
    nhan_x="số mã",
    dinh_dang_nhan="{:.0f}",
)

## 3 · Thời gian còn lại tới đáo hạn

In [8]:
con_han = con_han.assign(
    ngay_con_lai=(pd.to_datetime(con_han["maturity_date"]) - pd.Timestamp(HOM_NAY)).dt.days
)

print(con_han["ngay_con_lai"].describe().round(0).to_string())

phan_nhom = pd.cut(
    con_han["ngay_con_lai"],
    bins=[-1, 30, 90, 180, 365, 10_000],
    labels=["< 1 tháng", "1–3 tháng", "3–6 tháng", "6–12 tháng", "> 1 năm"],
)
dem = phan_nhom.value_counts().sort_index().reset_index()
dem.columns = ["thời gian còn lại", "số mã"]

bar_ngang(
    dem,
    nhan="thời gian còn lại",
    gia_tri="số mã",
    tieu_de="Chứng quyền theo thời gian còn lại tới đáo hạn",
    phu_de="Giá trị thời gian của chứng quyền tan rất nhanh trong tháng cuối",
    nhan_x="số mã",
    dinh_dang_nhan="{:.0f}",
)

count    311.0
mean     124.0
std       78.0
min        9.0
25%       49.0
50%      139.0
75%      184.0
max      323.0


## 4 · Moneyness — và một giới hạn của dữ liệu phải nói thẳng

**Moneyness** là quan hệ giữa giá cơ sở và giá thực hiện. Nó tính được từ dữ
liệu có sẵn:

```
moneyness = giá_cơ_sở_VND / exercise_price
```

⚠️ Nhưng **giá trị nội tại trên một chứng quyền thì KHÔNG**, vì nó cần **tỷ lệ
chuyển đổi** — bao nhiêu chứng quyền đổi được một cổ phiếu — và **cột đó không
có trong dữ liệu**:

In [9]:
print(f"Các cột `meta.warrants()` cung cấp: {list(client.meta.warrants().columns)}")
print("\n→ Không có cột tỷ lệ chuyển đổi (conversion ratio).")
print("  Nên các công thức dưới đây dừng ở MONEYNESS, không đi tới giá trị nội tại")
print("  hay premium. Tỷ lệ chuyển đổi nằm ở bản cáo bạch của từng đợt phát hành.")

Các cột `meta.warrants()` cung cấp: ['symbol', 'underlying', 'issuer', 'exercise_price', 'maturity_date']

→ Không có cột tỷ lệ chuyển đổi (conversion ratio).
  Nên các công thức dưới đây dừng ở MONEYNESS, không đi tới giá trị nội tại
  hay premium. Tỷ lệ chuyển đổi nằm ở bản cáo bạch của từng đợt phát hành.


Nói ra giới hạn này quan trọng hơn là bịa một công thức trông có vẻ đúng. Một
"premium" tính bằng tỷ lệ chuyển đổi mặc định 1:1 sẽ sai ở gần như mọi chứng
quyền trên thị trường Việt Nam, vì tỷ lệ thực tế thường là 2:1, 4:1, 10:1.

In [10]:
# Lấy giá cơ sở gần nhất cho mọi mã cơ sở có chứng quyền
ma_co_so = con_han["underlying"].unique().tolist()
gia_cs = client.eod.stock.ohlcv(ma_co_so, start=lui_ngay(HOM_NAY, ngay=10))

cs_gan_nhat = (
    gia_cs.sort_values("date").groupby("symbol", observed=True)["close"].last().rename("gia_cs_kvnd")
)
# ⚠️ Nhân 1.000: cổ phiếu là nghìn VND, exercise_price là VND thô
cs_vnd = (cs_gan_nhat * 1_000).rename("gia_cs_vnd")

phan_tich = con_han.merge(cs_vnd, left_on="underlying", right_index=True, how="inner")
phan_tich["moneyness"] = phan_tich["gia_cs_vnd"] / phan_tich["exercise_price"]
phan_tich["trang_thai"] = pd.cut(
    phan_tich["moneyness"],
    bins=[0, 0.9, 1.0, 1.1, 100],
    labels=["OTM sâu (<0,9)", "OTM (0,9–1,0)", "ITM (1,0–1,1)", "ITM sâu (>1,1)"],
)

print(phan_tich["trang_thai"].value_counts().sort_index().rename("số chứng quyền").to_frame().to_string())

                số chứng quyền
trang_thai                    
OTM sâu (<0,9)             169
OTM (0,9–1,0)               77
ITM (1,0–1,1)               43
ITM sâu (>1,1)              22


In [11]:
fig = go.Figure(
    go.Scatter(
        x=phan_tich["ngay_con_lai"],
        y=phan_tich["moneyness"],
        mode="markers",
        marker=dict(
            size=9,
            color=phan_tich["moneyness"],
            colorscale=[(0.0, GIAM), (0.5, "#f0efec"), (1.0, TANG)],
            cmid=1.0,
            opacity=0.8,
            line=dict(width=1, color="#fcfcfb"),
            colorbar=dict(title=dict(text="moneyness", font=dict(size=12, color="#52514e")),
                          thickness=12, outlinewidth=0),
        ),
        text=phan_tich["symbol"] + " · " + phan_tich["underlying"],
        hovertemplate="<b>%{text}</b><br>còn %{x} ngày<br>moneyness %{y:.3f}<extra></extra>",
    )
)
fig.add_hline(y=1.0, line_width=1, line_color="#898781", line_dash="dot",
              annotation_text="giá cơ sở = giá thực hiện", annotation_position="right")
fig.update_layout(
    title_text="Toàn bộ chứng quyền đang lưu hành: moneyness và thời gian còn lại<br>"
    "<sub style='color:#52514e'>Góc dưới bên trái là vùng nguy hiểm: OTM sâu và sắp đáo hạn</sub>",
    xaxis_title="số ngày tới đáo hạn",
    yaxis_title="moneyness (giá cơ sở ÷ giá thực hiện)",
    height=560,
)
fig

**Góc dưới bên trái đọc thế nào:** chứng quyền OTM sâu và chỉ còn vài tuần thì
gần như chắc chắn đáo hạn với giá trị bằng 0. Giá thị trường của chúng vẫn
dương — đó là phần giá trị thời gian còn sót lại, và nó tan rất nhanh.

In [12]:
nguy_hiem = phan_tich[(phan_tich["moneyness"] < 0.9) & (phan_tich["ngay_con_lai"] < 60)]
print(f"{len(nguy_hiem)}/{len(phan_tich)} chứng quyền đang OTM sâu và còn dưới 60 ngày")
if len(nguy_hiem):
    print(
        nguy_hiem.nsmallest(10, "moneyness")[
            ["symbol", "underlying", "issuer", "exercise_price", "gia_cs_vnd", "moneyness", "ngay_con_lai"]
        ]
        .round({"moneyness": 3})
        .to_string(index=False)
    )

46/311 chứng quyền đang OTM sâu và còn dưới 60 ngày
  symbol underlying issuer  exercise_price  gia_cs_vnd  moneyness  ngay_con_lai
CDGC2601        DGC  KISVN         76868.0     44150.0      0.574             9
CSHB2605        SHB  KISVN         19532.0     12000.0      0.614             9
CVRE2602        VRE  KISVN         37073.0     25300.0      0.682             9
CFPT2518        FPT    SSI        104449.0     71900.0      0.688            43
CSHB2603        SHB    VPX         17264.0     12000.0      0.695            23
CTCB2520        TCB    VND         44171.0     31150.0      0.705            28
CMSN2608        MSN  KISVN         95999.0     67700.0      0.705             9
CTPB2604        TPB  KISVN         20688.0     14700.0      0.711             9
CFPT2606        FPT    VPX        100614.0     71900.0      0.715            50
CFPT2602        FPT    TCX         98641.0     71900.0      0.729            41


## 5 · Thanh khoản — phần lớn chứng quyền gần như không giao dịch

In [13]:
MAU = con_han["symbol"].tolist()
gia_cw_tat = client.eod.warrant.ohlcv(MAU, start=lui_ngay(HOM_NAY, thang=1))

print(f"Lấy giá {gia_cw_tat['symbol'].nunique()}/{len(MAU)} chứng quyền")
print(f"Đơn vị: {[f'{k}={v}' for k, v in gia_cw_tat.attrs['finlens']['units'].items() if v]}")

tk = (
    gia_cw_tat.assign(gtgd=gia_cw_tat["close"] * gia_cw_tat["volume"])  # cả hai đều VND/đơn vị → VND
    .groupby("symbol", observed=True)
    .agg(gtgd_bq=("gtgd", "mean"), so_phien=("date", "count"),
         phien_khong_khop=("volume", lambda s: int((s == 0).sum())))
    .reset_index()
)
tk["ty_le_phien_khong_khop"] = (tk["phien_khong_khop"] / tk["so_phien"] * 100).round(0)

print(f"\nGTGD bình quân (triệu VND/phiên): trung vị {tk['gtgd_bq'].median() / 1e6:,.1f}, "
      f"cao nhất {tk['gtgd_bq'].max() / 1e6:,.0f}")
print(f"Số mã có GTGD bình quân dưới 100 triệu/phiên: "
      f"{(tk['gtgd_bq'] < 100e6).sum()}/{len(tk)} ({(tk['gtgd_bq'] < 100e6).mean():.0%})")
print(f"Số mã có ≥ 1 phiên không khớp lệnh nào: "
      f"{(tk['phien_khong_khop'] > 0).sum()}/{len(tk)} ({(tk['phien_khong_khop'] > 0).mean():.0%})")

Lấy giá 311/311 chứng quyền
Đơn vị: ['open=VND', 'high=VND', 'low=VND', 'close=VND', 'volume=share']

GTGD bình quân (triệu VND/phiên): trung vị 55.4, cao nhất 1,802
Số mã có GTGD bình quân dưới 100 triệu/phiên: 192/311 (62%)
Số mã có ≥ 1 phiên không khớp lệnh nào: 88/311 (28%)


In [14]:
top_tk = tk.nlargest(15, "gtgd_bq").assign(gtgd_ty=lambda d: (d["gtgd_bq"] / 1e9).round(2))

bar_ngang(
    top_tk,
    nhan="symbol",
    gia_tri="gtgd_ty",
    tieu_de="15 chứng quyền thanh khoản nhất",
    phu_de="Giá trị giao dịch bình quân một tháng · phần còn lại của thị trường mỏng hơn nhiều",
    nhan_x="tỷ đồng/phiên",
    dinh_dang_nhan="{:.2f}",
)

⚠️ **Thanh khoản là rủi ro chính của chứng quyền, không phải biến động giá.**
Một chứng quyền có vài phiên không khớp lệnh nào nghĩa là có những ngày bạn
**không thoát ra được ở bất kỳ giá nào**. Với một sản phẩm có ngày hết hạn cố
định, đó không phải bất tiện — đó là mất trắng.

## 6 · Chứng quyền so với cơ sở — đòn bẩy đi cả hai chiều

In [15]:
MA_CS = phan_tich["underlying"].value_counts().index[0]
ung_vien = (
    phan_tich[phan_tich["underlying"] == MA_CS]
    .merge(tk, on="symbol")
    .nlargest(3, "gtgd_bq")["symbol"]
    .tolist()
)
print(f"Cơ sở {MA_CS}, ba chứng quyền thanh khoản nhất: {ung_vien}")

cw3 = client.eod.warrant.ohlcv(ung_vien, start=lui_ngay(HOM_NAY, thang=3))
cs3 = client.eod.stock.ohlcv(MA_CS, start=lui_ngay(HOM_NAY, thang=3))

# Chuẩn hoá về gốc 100 — đây là cách DUY NHẤT đặt hai đơn vị cạnh nhau cho đúng
def goc_100(df: pd.DataFrame, khoa: str = "symbol") -> pd.DataFrame:
    df = df.sort_values([khoa, "date"])
    dau = df.groupby(khoa, observed=True)["close"].transform("first")
    return df.assign(chi_so_100=df["close"] / dau * 100)


so_sanh = pd.concat([goc_100(cw3), goc_100(cs3)])

duong(
    so_sanh,
    x="date",
    y="chi_so_100",
    theo="symbol",
    tieu_de=f"{MA_CS} và ba chứng quyền của nó — chuẩn hoá về 100",
    phu_de="Chuẩn hoá là cách duy nhất đặt VND thô cạnh nghìn VND mà không sai",
    nhan_y="chỉ số (gốc = 100)",
)

Cơ sở HPG, ba chứng quyền thanh khoản nhất: ['CHPG2628', 'CHPG2625', 'CHPG2627']


In [16]:
bien_dong = (
    so_sanh.sort_values(["symbol", "date"])
    .assign(ls=lambda d: d.groupby("symbol", observed=True)["close"].pct_change() * 100)
    .groupby("symbol", observed=True)
    .agg(bien_dong_ngay=("ls", "std"), thay_doi_ky=("chi_so_100", "last"))
)
bien_dong["thay_doi_ky"] = (bien_dong["thay_doi_ky"] - 100).round(1)
bien_dong["bien_dong_ngay"] = bien_dong["bien_dong_ngay"].round(2)
bien_dong["so_voi_co_so"] = (bien_dong["bien_dong_ngay"] / bien_dong.loc[MA_CS, "bien_dong_ngay"]).round(1)
bien_dong.rename(
    columns={"bien_dong_ngay": "độ lệch chuẩn ngày %", "thay_doi_ky": "thay đổi cả kỳ %",
             "so_voi_co_so": "biến động gấp cơ sở (lần)"}
)

,độ lệch chuẩn ngày %,thay đổi cả kỳ %,biến động gấp cơ sở (lần)
symbol,,,
CHPG2625,7.56,-61.0,5.0
CHPG2627,11.08,-14.0,7.3
CHPG2628,8.69,-28.6,5.7
HPG,1.52,-10.3,1.0


Cột cuối là đòn bẩy thực tế đo được. Nó **đi cả hai chiều** — và với chứng
quyền còn thêm hai thứ cổ phiếu không có: giá trị thời gian tan dần mỗi ngày,
và một ngày hết hạn mà sau đó mọi thứ về 0 nếu OTM.

## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Danh mục chứng quyền còn hạn | `meta.warrants()` |
| Kể cả đã đáo hạn | `meta.warrants(active_only=False)` |
| Chứng quyền của một cơ sở | `meta.warrants(underlying="HPG")` |
| Giá chứng quyền | `eod.warrant.ohlcv("CHPG2525")` — **VND thô** |

**Bốn điều đáng nhớ:**

1. ⚠️ **Giá chứng quyền là VND thô, giá cổ phiếu là nghìn VND.** Ghép chúng mà
   không quy đổi là sai 1000 lần, không exception. Viết một hàm **đọc đơn vị
   từ `attrs`** thay vì nhân 1.000 theo trí nhớ.
2. **Dữ liệu không có tỷ lệ chuyển đổi**, nên tính được moneyness nhưng không
   tính được giá trị nội tại hay premium. Nói ra giới hạn đó tốt hơn là bịa một
   tỷ lệ 1:1 sai ở gần như mọi mã.
3. **Thanh khoản là rủi ro chính.** Phần lớn chứng quyền có GTGD bình quân dưới
   100 triệu/phiên và có phiên không khớp lệnh nào.
4. Muốn đặt chứng quyền cạnh cổ phiếu trên một biểu đồ thì **chuẩn hoá về gốc
   100** — không có cách nào khác đúng.

---

**Tiếp theo:** [`44_vi_mo_va_thi_truong.ipynb`](44_vi_mo_va_thi_truong.ipynb) —
notebook cuối: dashboard vĩ mô đặt cạnh VNINDEX.